# G1 overnight training (run with *Save Version → Save & Run All*)

Settings: **Accelerator = GPU T4 x2**, **Internet = On**, **Persistence = Files**. Set them for the *version* too (Save Version → Advanced).

Runs ~5.5–6 h for the full 250M steps and stops cleanly before Kaggle's 12 h limit (`MAX_HOURS`). When it finishes, open the version's **Output** tab: `g1_rl/outputs/runs/<RUN_NAME>/` holds `training_curves.png`, `video_walk_forward.mp4`, `policy_final.npz/.json`, checkpoints and `metrics.jsonl`.

**To continue a run in a new version:** Add Input → *Your Work* → this notebook's previous version (its output), set `RESUME = True`, keep the same `RUN_NAME`.

In [ ]:
RUN_NAME  = "g1_walk_v1"
MAX_HOURS = 11.0        # stop (with checkpoint) before Kaggle's 12 h limit
RESUME    = False       # True: continue RUN_NAME from an attached previous version's output
EXTRA     = ""          # e.g. "--total-steps 400e6" 

In [ ]:
import os, re, shutil, glob, subprocess
# ---- where to get the code -------------------------------------------------
SOURCE  = "github"       # "github" or "dataset"
GH_REPO = "plant7-arch/g1_rl"
GIT_REF = "v0.5.1"       # a version tag, or "main" for the latest push
# Private repo: add a Kaggle Secret named GITHUB_TOKEN (Add-ons -> Secrets), read-only token.
# ----------------------------------------------------------------------------
WORK = "/kaggle/working/g1_rl"

def pkg_version(root):
    m = re.search(r'__version__\s*=\s*"([\d.]+)"', open(f"{root}/g1_rl/__init__.py").read())
    return tuple(int(x) for x in m.group(1).split(".")) if m else (0,)

if SOURCE == "github":
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None                       # public repo, or secret not attached
    auth = f"x-access-token:{token}@" if token else ""
    src = "/kaggle/tmp/g1_rl_src"
    shutil.rmtree(src, ignore_errors=True)
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", GIT_REF,
                        f"https://{auth}github.com/{GH_REPO}.git", src],
                       capture_output=True, text=True)
    if r.returncode:                        # never print the URL (it may contain the token)
        raise SystemExit("git clone failed: " + r.stderr.replace(token or "\0", "***")[-500:])
    sha = subprocess.run(["git", "-C", src, "rev-parse", "--short", "HEAD"],
                         capture_output=True, text=True).stdout.strip()
    print(f"cloned {GH_REPO}@{GIT_REF} ({sha})")
else:
    roots = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
             if 'name = "g1_rl"' in open(p).read()]
    assert roots, "Attach the g1_rl dataset or use SOURCE='github'"
    roots.sort(key=pkg_version, reverse=True)
    for r in roots:
        print(f"found {'.'.join(map(str, pkg_version(r)))}  {r}")
    if len(roots) > 1 and pkg_version(roots[0]) == pkg_version(roots[1]):
        raise SystemExit("Two copies with the same version: delete the old zip from the dataset.")
    src = roots[0]

# Replace the code but keep outputs/ (checkpoints, logs, videos) from earlier sessions.
os.makedirs(WORK, exist_ok=True)
for item in os.listdir(WORK):
    if item != "outputs":
        p = os.path.join(WORK, item)
        shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
shutil.copytree(src, WORK, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".git"))
os.chdir(WORK)
print("g1_rl version", ".".join(map(str, pkg_version(WORK))), "|", sorted(os.listdir()))

In [ ]:
!bash scripts/install_kaggle.sh

In [ ]:
import glob, os, subprocess
resume_arg = ""
if RESUME:
    hits = sorted(glob.glob(f"/kaggle/input/**/outputs/runs/{RUN_NAME}/ckpt/latest.txt", recursive=True))
    hits += sorted(glob.glob(f"outputs/runs/{RUN_NAME}/ckpt/latest.txt"))
    assert hits, f"RESUME=True but no checkpoint for {RUN_NAME} under /kaggle/input or outputs/"
    resume_arg = "--resume " + os.path.dirname(os.path.dirname(hits[-1]))
    print("resuming from", resume_arg)
cmd = f"python scripts/train.py --preset full --run-name {RUN_NAME} --max-hours {MAX_HOURS} {resume_arg} {EXTRA}"
print(cmd)
rc = subprocess.call(cmd, shell=True)
print("exit code", rc)

In [ ]:
!python scripts/plot_metrics.py outputs/runs/{RUN_NAME} --show-terms
!cat outputs/runs/{RUN_NAME}/summary.json
!ls -la outputs/runs/{RUN_NAME}